# Ch. 3 · TP1 Régression scientifique et validation

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch03-tp1-regression.ipynb)

**IA pour les sciences — M1.** Baseline, validation croisée, complexité, résidus et extrapolation, puis un jeu de données réel.

**Durée indicative :** 3 h

**Objectifs**

- Comparer des modèles de régression à une baseline avec MAE et RMSE.
- Visualiser le compromis biais–variance en faisant varier la complexité.
- Diagnostiquer un modèle par ses résidus.
- Montrer qu’un modèle flexible peut échouer en extrapolation.
- Appliquer le protocole à un jeu réel (diabetes, scikit-learn).

**Prérequis :** chapitre 2 ; notions de régression du cours du chapitre 3.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 1. Données simulées : réponse d’un instrument

L’équation génératrice $y = 1{,}5 + 0{,}8T + 0{,}015T^2 + \varepsilon$, $\varepsilon\sim\mathcal{N}(0, 7^2)$, sert à **contrôler** l’expérience :
nous connaissons la vérité, ce qui est rare en recherche. Le bruit $\sigma=7$ fixe l’erreur minimale atteignable (≈ 5,6 en MAE).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 180
T = rng.uniform(10, 75, n)
y = 1.5 + 0.8*T + 0.015*T**2 + rng.normal(0, 7, n)
df = pd.DataFrame({'temperature_C': T, 'signal_u.a.': y})
print(df.describe().round(2))
df.plot.scatter(x='temperature_C', y='signal_u.a.', alpha=.7, title='Réponse simulée d’un instrument')
plt.show()

## 2. Séparer, entraîner, comparer à une référence

- $MAE = \frac1n\sum|y_i-\hat y_i|$ (unité de $y$, robuste) ; $RMSE = \sqrt{\frac1n\sum(y_i-\hat y_i)^2}$ (pénalise les grosses erreurs).
- La **baseline** prédit la moyenne de l’entraînement : un modèle qui ne la bat pas n’a rien appris d’utile.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

X = df[['temperature_C']]
y = df['signal_u.a.']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
models = {
    'Référence': DummyRegressor(strategy='mean'),
    'Linéaire': LinearRegression(),
    'Forêt': RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f'{name:10s} MAE = {mean_absolute_error(y_test, pred):6.2f}   RMSE = {rmse(y_test, pred):6.2f}')

## 3. Démonstration : complexité, biais et variance

Nous ajustons des polynômes de degré 1 à 15. L’erreur d’**entraînement** baisse toujours ; l’erreur de **validation croisée**
baisse, puis remonte : c’est le **surapprentissage**. On choisit la complexité au minimum de la courbe de validation,
**sans toucher au test**.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

degres = range(1, 16)
err_train, err_cv = [], []
for d in degres:
    pipe = make_pipeline(StandardScaler(), PolynomialFeatures(d), LinearRegression())
    pipe.fit(X_train, y_train)
    err_train.append(mean_absolute_error(y_train, pipe.predict(X_train)))
    err_cv.append(-cross_val_score(pipe, X_train, y_train, cv=5,
                                   scoring='neg_mean_absolute_error').mean())

plt.plot(degres, err_train, 'o-', label='Entraînement')
plt.plot(degres, err_cv, 's-', label='Validation croisée (5 plis)')
plt.axhline(7 * np.sqrt(2/np.pi), color='gray', ls=':', label='Bruit irréductible (MAE théorique)')
plt.xlabel('Degré du polynôme'); plt.ylabel('MAE (u.a.)'); plt.yscale('log')
plt.title('Compromis biais–variance'); plt.legend(); plt.show()
print('Degré retenu par validation croisée :', degres[int(np.argmin(err_cv))])

## 4. Diagnostiquer par les résidus

Un bon modèle laisse des résidus $y-\hat y$ **sans structure** : centrés sur 0, de dispersion constante.
Une courbure résiduelle signale un terme manquant (ici le terme quadratique pour le modèle linéaire).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, name in zip(axes, ['Linéaire', 'Forêt']):
    p = models[name].predict(X_test)
    ax.scatter(p, y_test - p, alpha=.7)
    ax.axhline(0, color='black', ls='--')
    ax.set(title=f'Résidus — {name}', xlabel='Prédiction (u.a.)', ylabel='Résidu (u.a.)')
plt.tight_layout(); plt.show()

scores = cross_val_score(models['Forêt'], X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f'Forêt — MAE CV : {-scores.mean():.2f} ± {scores.std():.2f}')

### ✏️ Exercice 1 — Extrapolation : la forêt hors de son domaine

Entraînez les modèles **uniquement sur T < 50 °C** et testez-les sur **T ≥ 50 °C**. Comparez la MAE de la régression linéaire, d’un polynôme de degré 2 et de la forêt. Tracez les prédictions sur toute la plage.

In [ ]:
masque = df['temperature_C'] < 50
X_in, y_in = X[masque], y[masque]
X_out, y_out = X[~masque], y[~masque]
candidats = {
    'Linéaire': LinearRegression(),
    'Polynôme 2': make_pipeline(PolynomialFeatures(2), LinearRegression()),
    'Forêt': RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42),
}
grille = pd.DataFrame({'temperature_C': np.linspace(10, 75, 200)})
plt.scatter(X['temperature_C'], y, c=np.where(masque, 'C0', 'lightgray'), s=12)
for name, m in candidats.items():
    m.fit(..., ...)
    print(name, 'MAE hors domaine =', round(mean_absolute_error(y_out, ...), 2))
    plt.plot(grille['temperature_C'], m.predict(grille), label=name)
plt.axvline(50, color='k', ls=':'); plt.legend(); plt.show()

<details><summary><b>Voir le corrigé</b></summary>

```python
masque = df['temperature_C'] < 50
X_in, y_in = X[masque], y[masque]
X_out, y_out = X[~masque], y[~masque]
candidats = {
    'Linéaire': LinearRegression(),
    'Polynôme 2': make_pipeline(PolynomialFeatures(2), LinearRegression()),
    'Forêt': RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42),
}
grille = pd.DataFrame({'temperature_C': np.linspace(10, 75, 200)})
plt.scatter(X['temperature_C'], y, c=np.where(masque, 'C0', 'lightgray'), s=12)
for name, m in candidats.items():
    m.fit(X_in, y_in)
    print(name, 'MAE hors domaine =', round(mean_absolute_error(y_out, m.predict(X_out)), 2))
    plt.plot(grille['temperature_C'], m.predict(grille), label=name)
plt.axvline(50, color='k', ls=':'); plt.legend(); plt.show()
```

Une forêt prédit des **moyennes de feuilles** : au-delà de 50 °C, sa prédiction reste plate et l’erreur triple. Les modèles paramétriques prolongent une tendance et font bien mieux, mais même le polynôme de degré 2, qui a la bonne forme, voit son erreur presque doubler : la courbure, estimée sur un domaine étroit, est imprécise. En recherche, on ne connaît pas la vraie loi : il faut toujours déclarer le **domaine de validité** d’un modèle.

</details>

## 5. Jeu réel : progression du diabète (scikit-learn)

442 patients, 10 variables cliniques déjà centrées-réduites (âge, IMC, pression artérielle, 6 mesures sanguines),
cible = mesure quantitative de progression de la maladie un an après. C’est un jeu **réel**, bruité, où le gain des modèles est modeste.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_validate

Xd, yd = load_diabetes(return_X_y=True, as_frame=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=0)
cv = KFold(n_splits=5, shuffle=True, random_state=0)
candidats = {
    'Référence': DummyRegressor(),
    'Linéaire': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Forêt': RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0),
}
lignes = []
for name, m in candidats.items():
    r = cross_validate(m, Xd_tr, yd_tr, cv=cv, scoring=['neg_mean_absolute_error', 'r2'])
    lignes.append({'modèle': name,
                   'MAE CV': -r['test_neg_mean_absolute_error'].mean(),
                   'écart-type': r['test_neg_mean_absolute_error'].std(),
                   'R² CV': r['test_r2'].mean()})
print(pd.DataFrame(lignes).round(3).to_string(index=False))

### ✏️ Exercice 2 — Régularisation Ridge

Faites varier `alpha` de $10^{-3}$ à $10^{3}$ (`np.logspace(-3, 3, 13)`) et tracez la MAE de validation croisée. Évaluez ensuite **une seule fois** sur le test le meilleur modèle.

In [ ]:
alphas = np.logspace(-3, 3, 13)
mae = []
for a in alphas:
    s = cross_val_score(Ridge(alpha=a), Xd_tr, yd_tr, cv=cv, scoring='neg_mean_absolute_error')
    mae.append(...)
meilleur = alphas[int(np.argmin(mae))]
plt.semilogx(alphas, mae, 'o-'); plt.xlabel('alpha'); plt.ylabel('MAE CV'); plt.show()
final = Ridge(alpha=meilleur).fit(..., ...)
print('alpha retenu :', meilleur, '— MAE test :', round(mean_absolute_error(yd_te, ...), 2))

<details><summary><b>Voir le corrigé</b></summary>

```python
alphas = np.logspace(-3, 3, 13)
mae = []
for a in alphas:
    s = cross_val_score(Ridge(alpha=a), Xd_tr, yd_tr, cv=cv, scoring='neg_mean_absolute_error')
    mae.append(-s.mean())
meilleur = alphas[int(np.argmin(mae))]
plt.semilogx(alphas, mae, 'o-'); plt.xlabel('alpha'); plt.ylabel('MAE CV'); plt.show()
final = Ridge(alpha=meilleur).fit(Xd_tr, yd_tr)
print('alpha retenu :', meilleur, '— MAE test :', round(mean_absolute_error(yd_te, final.predict(Xd_te)), 2))
```

Ici, les variables sont déjà mises à une très petite échelle : `alpha=1` régularise beaucoup trop, d’où le choix d’un petit `alpha`. Ridge minimise $\sum(y_i-\hat y_i)^2 + \alpha\|\beta\|^2$ : un grand $\alpha$ rétrécit les coefficients (plus de biais, moins de variance). Le test n’intervient qu’**après** le choix de $\alpha$.

</details>

## Interprétation scientifique (à rédiger)

1. Quel gain chaque modèle apporte-t-il par rapport à la baseline, avec quelle incertitude ?
2. Que révèlent les résidus du modèle linéaire sur les données simulées ?
3. Quel modèle choisiriez-vous pour extrapoler, et à quelle condition ?
4. Sur le jeu diabetes, un R² ≈ 0,5 est-il « mauvais » ? Que faudrait-il pour un usage clinique ?